# Recorrido explicado — El sistema de talleres de Circo Bus

Este notebook acompaña una explicación de los temas del repaso con un caso común y ejemplos resueltos. Las celdas de código calculan métricas y muestran mecanismos en una versión pequeña.

Los datos son inventados. El código usa solo Python estándar y no llama servicios externos. Ejecutá las celdas en orden para ver cada demostración.

## Un sistema, tres capacidades

Circo Bus estima cuántas personas asistirán a sus talleres para organizar lugares y materiales. El asistente consulta manuales de seguridad y conserva datos relevantes de cada conversación. Un agente puede consultar o modificar los turnos de los equipos.

El recorrido conecta tres temas: **predicción y monitoreo → consulta de documentos y memoria → acción con controles y registro**.

## 1. Aprender a partir de ejemplos

**Aprendizaje supervisado**: el modelo aprende con ejemplos que incluyen las entradas y el resultado correcto conocido.

**Aprendizaje no supervisado**: busca grupos o patrones sin que cada ejemplo tenga una respuesta correcta indicada.

**Aprendizaje por refuerzo**: un agente aprende qué acciones convienen a partir de recompensas o resultados.

La **regresión** es un problema supervisado cuyo resultado es un número. En Circo Bus, las variables predictoras son sede, día y horario; la etiqueta o resultado objetivo es la asistencia real. El sistema aprende la relación entre esas entradas y ese resultado.

### Ejemplo resuelto

La tabla compara cinco días. La diferencia absoluta es la distancia entre la cantidad estimada y la cantidad real, sin importar si el modelo quedó corto o largo. En el cuarto día se ve un error grande que va a influir más en el RMSE.

In [ ]:
asistencia_real = [18, 21, 15, 32, 27]
asistencia_estimada = [20, 19, 14, 22, 30]

print("Día | Real | Estimada")
for dia, (real, estimada) in enumerate(
    zip(asistencia_real, asistencia_estimada), start=1
):
    print(f"{dia:>3} | {real:>4} | {estimada:>8}")

### Leer el error del modelo

El **MAE** promedia el tamaño absoluto de los errores. Se expresa en personas y es fácil de explicar.

El **RMSE** también se expresa en personas, pero da más peso a errores grandes. Puede servir cuando una estimación muy alejada causa un problema serio.

El **R²** compara el modelo con una referencia simple: estimar el promedio de asistencia para todos los casos. No dice cuántas personas de más o de menos estimó. Ninguna de estas métricas, por sí sola, indica si el problema fue sobreestimar o subestimar.

In [ ]:
from math import sqrt

errores = [
    estimada - real
    for real, estimada in zip(asistencia_real, asistencia_estimada)
]

mae = sum(abs(error) for error in errores) / len(errores)
rmse = sqrt(sum(error ** 2 for error in errores) / len(errores))

promedio_real = sum(asistencia_real) / len(asistencia_real)
suma_error_modelo = sum(error ** 2 for error in errores)
suma_error_referencia = sum(
    (real - promedio_real) ** 2 for real in asistencia_real
)
r2 = 1 - suma_error_modelo / suma_error_referencia

print(f"MAE: {mae:.2f} personas")
print(f"RMSE: {rmse:.2f} personas")
print(f"R²: {r2:.2f}")

### Lectura de este resultado

En esta muestra, el error absoluto promedio es de **3,6 personas**. El RMSE es mayor que el MAE porque el error de diez personas pesa más al elevarlo al cuadrado. El R² compara el modelo con la referencia del promedio; no significa que el modelo acierte ese porcentaje de veces.

Son resultados de cinco datos inventados. No prueban que el modelo de Circo Bus sea útil en otras sedes ni permiten saber por qué aparece un error. Si quedarse corto cuesta distinto que tener materiales de más, también se considera la dirección de cada error y sus consecuencias.

## 2. Cambios en los datos y en el rendimiento

La **deriva de datos** (*data drift*) ocurre cuando cambia la distribución de las entradas, por ejemplo, cuando aumentan los talleres de fin de semana. La **deriva de concepto** ocurre cuando cambia la relación entre las entradas y el resultado: las mismas condiciones ya no producen patrones de asistencia parecidos.

La **degradación del rendimiento** se confirma al comparar predicciones con resultados reales y ver que el error aumentó. Un cambio en la distribución de las entradas es una señal para investigar; por sí solo no confirma una pérdida de precisión.

El ciclo de vida incluye revisar la calidad de los datos, medir el rendimiento cuando llegan resultados reales, investigar los cambios, evaluar una posible actualización y seguir monitoreando después.

### Ejemplo: cambió la composición de los talleres

El siguiente conteo compara qué proporción de talleres se dicta durante el fin de semana. Sirve para ver una diferencia en las entradas entre los registros históricos y los recientes. Todavía no indica si el modelo predice peor: para medir eso hacen falta las asistencias reales.

In [ ]:
historico = {"entre_semana": 95, "fin_de_semana": 5}
reciente = {"entre_semana": 65, "fin_de_semana": 35}

for periodo, registros in [("Histórico", historico), ("Reciente", reciente)]:
    total = sum(registros.values())
    porcentaje_fin_de_semana = registros["fin_de_semana"] / total * 100
    print(f"{periodo}: {porcentaje_fin_de_semana:.0f}% de fin de semana")

La proporción de talleres de fin de semana pasó del 5 % al 35 %. La coordinación puede revisar si ese tipo de jornada estaba representado durante el entrenamiento y medir el error cuando ya conozca las asistencias reales.

Si se actualiza el modelo, la nueva versión se evalúa con datos recientes y representativos. La validación permite elegir entre alternativas; los datos de prueba se reservan para el informe final. Después de actualizar, el monitoreo continúa.

## 3. Conocimiento del asistente y memoria

Un **LLM** es un modelo que procesa y genera texto. El texto se procesa en unidades llamadas **tokens**; la cantidad de tokens afecta el costo y el espacio limitado que el modelo puede considerar en una consulta, llamado **ventana de contexto**.

La **memoria conversacional** conserva información útil de una sesión, como “solo puedo después de las 17”. Debe asociarse con la conversación correcta. No es una biblioteca de manuales.

**RAG** significa generación aumentada por recuperación. Antes de responder, el sistema busca información pertinente en documentos y agrega fragmentos seleccionados al contexto del LLM. El flujo típico tiene tres partes: preparar documentos, recuperar fragmentos y generar una respuesta fundamentada en ellos.

### Memoria de una conversación

Este diccionario representa el estado asociado a una sesión. En la conversación CB-014, la persona indicó que está disponible después de las 17. El identificador evita mezclar ese dato con otra sesión.

El diccionario vive solo mientras se ejecuta el kernel. Una aplicación real necesita almacenar la memoria de forma persistente y recuperarla cuando continúa la misma conversación.

In [ ]:
memoria_por_sesion = {
    "CB-014": {"preferencia_horaria": "después de las 17"}
}

sesion_actual = "CB-014"
print(memoria_por_sesion[sesion_actual]["preferencia_horaria"])

### Cómo funciona RAG con los manuales

Un documento se divide en **fragmentos** (*chunks*) que conservan datos como el título, la sección y la versión. Un **embedding** representa el texto con números para comparar semejanzas. Un **índice vectorial** organiza esos embeddings para recuperar pasajes cercanos a una consulta.

El siguiente ejemplo usa coincidencia de palabras para hacer visible la selección de una fuente sin descargar modelos. Esta demostración simplificada no calcula embeddings, no reconoce sinónimos y no es un sistema RAG completo.

In [ ]:
documentos = [
    {
        "fuente": "Manual del torno, sección 2",
        "texto": "Antes de encender el torno, colocate los anteojos de protección."
    },
    {
        "fuente": "Guía de reservas, sección 1",
        "texto": "Los turnos de equipos se reservan por bloques de una hora."
    },
    {
        "fuente": "Manual de impresora 3D, sección 4",
        "texto": "La impresión larga debe iniciarse con la tapa cerrada."
    }
]

consulta = "¿Qué protección necesito para usar el torno?"
palabras_vacias = {"el", "la", "los", "de", "para", "que", "con", "un", "una", "y"}

def palabras(texto):
    return {
        palabra.strip(".,;:¿?¡!").lower()
        for palabra in texto.split()
        if palabra.strip(".,;:¿?¡!").lower() not in palabras_vacias
    }

consulta_palabras = palabras(consulta)
mejor_documento = max(
    documentos,
    key=lambda documento: len(consulta_palabras & palabras(documento["texto"]))
)

print("Fuente recuperada:", mejor_documento["fuente"])
print("Fragmento:", mejor_documento["texto"])

La búsqueda encontró una parte del manual del torno. Una respuesta basada en esta evidencia indica que hay que colocarse anteojos de protección antes de encenderlo e identifica la fuente utilizada.

En un RAG real, un mecanismo de búsqueda selecciona fragmentos y un LLM prepara una respuesta con ese contexto. El asistente informa la limitación si no encuentra evidencia pertinente o si las fuentes se contradicen.

## 4. Agentes, herramientas y supervisión

Un **agente** puede decidir pasos para alcanzar un objetivo e invocar **herramientas**, como consultar o modificar la cola de turnos. **ReAct** es un patrón que intercala pasos de razonamiento y acciones con herramientas para observar resultados y continuar.

Una **inyección de prompts** es una instrucción engañosa dentro de un mensaje o archivo que intenta desviar al modelo. En este caso, el archivo pide borrar trabajos, pero el contenido de ese archivo no autoriza la acción.

**Human-in-the-Loop (HITL)** incorpora una revisión humana antes de una decisión o acción definida. La revisión recibe la propuesta concreta, su alcance y sus consecuencias. La aplicación mantiene además los permisos técnicos y las validaciones.

### Una propuesta en espera de revisión

El ejemplo deja la cancelación en estado pendiente. La herramienta todavía no cambió la cola y el resultado queda registrado como “no ejecutado”. Si la persona rechaza la propuesta o no responde, la acción permanece sin ejecutar.

In [ ]:
import json

registro = {
    "session_id": "CB-027",
    "tool": "cancelar_trabajos",
    "decision": "pausar y pedir revisión",
    "human_review": "pendiente",
    "result": "no ejecutado"
}

print(json.dumps(registro, ensure_ascii=False, indent=2))

### Trazabilidad e ISO/IEC 42001

La **trazabilidad** permite reconstruir qué ocurrió: sesión, herramienta, propuesta, decisión, revisión y resultado. Un registro JSON organiza esos campos para que personas y aplicaciones puedan consultarlos.

**ISO/IEC 42001:2023** establece requisitos para que una organización gestione sus sistemas de IA y mejore ese sistema de gestión. Un registro aporta evidencia para revisar un evento; no prueba por sí solo que la decisión fue correcta ni que la organización cumple la norma.

Los permisos y las validaciones deben comprobarse en la aplicación y en la herramienta. La intervención humana no reemplaza esos controles.

## Ideas para recordar

- La regresión estima un valor numérico con ejemplos que incluyen resultados conocidos.
- La deriva de entradas y el aumento del error son señales distintas; se relacionan al comparar los datos con resultados reales.
- RAG recupera información de manuales; la memoria sostiene el contexto de una conversación.
- ReAct organiza el uso de herramientas; HITL agrega una revisión en un punto definido.
- Los permisos limitan acciones y la trazabilidad deja evidencia para reconstruir lo ocurrido.

### Fuentes

- [ISO/IEC 42001:2023 — ISO](https://www.iso.org/standard/81230.html)
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- [OWASP — LLM01: Prompt Injection](https://genai.owasp.org/llmrisk/llm01-prompt-injection/)